# NMC data QC 

# Import Required packages

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
import anndata as ad
import seaborn as sns
import scanpy as sc
import yaml
%matplotlib inline
%config InlineBackend.figure_format='retina'

# Required Inputs

In [2]:
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Set up directories
metadata_dir = Path(config['metadata_dir'])
preprocessed_dir = Path(config['preprocessed_dir'])

# make a roi list of directories in preprocessed_dir that are not .DS_Store
roi_list = [roi for roi in preprocessed_dir.iterdir() if roi.is_dir() and roi.name != '.DS_Store']


# Extract patients metadata and export it to the new format

In [3]:
patients_metadata = pd.read_excel(metadata_dir / 'patient_metadata.xlsx')
patients_metadata["outcome"] = patients_metadata["outcome"].str.extract('(\d+)').astype(int)
#remove excluded rois
patients_metadata = patients_metadata.loc[~patients_metadata["rois"].isna()]
#add a new column if the  "outcome" column is greater than 9
patients_metadata["outcome_group"] = np.where(patients_metadata["outcome"] > 6, "long_term survival", "short_term survival")
patients_metadata["rois"] = patients_metadata["rois"].astype(str)
patients_metadata['rois'] = patients_metadata['rois'].str.split(r'\+ ')
patients_metadata = patients_metadata.explode('rois', ignore_index=True)
patients_metadata['rois'] = patients_metadata['rois'].str.strip()
for i in range(len(patients_metadata)):
    if patients_metadata['rois'][i].find('-') != -1:
        new_patient = patients_metadata['rois'][i].split('-')
        new_patient = np.array(range(int(new_patient[0]), int(new_patient[1])+1))
        patients_metadata['rois'][i] = new_patient

patients_metadata = patients_metadata.explode('rois', ignore_index=True)
patients_metadata["rois"] = patients_metadata["rois"].astype(str)
patients_metadata["exp_name"] = "Run" +  patients_metadata['run'].astype(int).astype(str) + '_ROI' + patients_metadata['rois'].astype(str)
patients_metadata.to_csv(metadata_dir / 'new_patients_metadata.csv', index=False)

/var/folders/zj/m3tnl76s7d5894m0ndctg1kh0000gn/T/ipykernel_1820/757644536.py:15: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  patients_metadata['rois'][i] = new_patient
/var/folders/zj/m3tnl76s7d5894m0ndctg1kh0000gn/T/ipykernel_1820/7576445

## Compare number of ROIS based on the survival time

# Check the Cell type Distribution

## Remove the trash cells

In [6]:
for roi in roi_list:
    exp = pd.read_csv(roi / 'exp.csv')
    meta = pd.read_csv(roi / 'metadata.csv')
    if "Trash" not in meta["cell_type"].unique():
        print("No trash cells in metadata")
        continue
    else:
        pass
    print("Removing trash cells")
    meta = meta.loc[meta["cell_type"] != "Trash"]
    exp = exp[exp["cell_id"].isin(meta.cell_id)]
    #export csvs
    exp.to_csv(roi / 'exp.csv', index=False)
    meta.to_csv(roi / 'metadata.csv', index=False)

No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata


## Extract the names of all cell types

In [5]:
# Create a list of all cell types
all_cell_types = []  
for i, roi in enumerate(roi_list):
    meta = pd.read_csv(roi / 'metadata.csv')
    all_cell_types.extend(meta["cell_type"].unique())
# Create a set of unique cell types
all_cell_types = sorted(set(all_cell_types))   
# Create a DataFrame with all cell types and initialize counts to zero
meta = pd.read_csv(roi_list[0] / 'metadata.csv')
meta_counts = meta["cell_type"].value_counts()
for cell_type in all_cell_types:
        if cell_type not in meta_counts:
            meta_counts[cell_type] = 0
sorted_order = meta_counts.index  # Extract the index order from a reference DataFrame
all_cell_types = list(sorted_order)
all_cell_types

['Tumor_cells',
 'Actin+_cells',
 'NFC',
 'Endothelial_cells',
 'CD4+_T_cells',
 'M1_macrophages',
 'DCs',
 'M1_M2_macrophages',
 'Mast_cells',
 'M2_macrophages',
 'CD8+_T_cells',
 'Treg_T_cells',
 'Plasma_cells',
 'Lymphatic_endothelial_cells',
 'MDSCs',
 'B_cells',
 'Granulocytes',
 'Epithelial_cells',
 'NK_cells']

## Plot cell types distribution

## Checking if the number of antibodies is equal in all datasets

In [4]:
for roi in roi_list:
    exp = pd.read_csv(roi / 'exp.csv')
    print (roi.name)
    print(len(exp.columns))

Run141_ROI11
87
Run141_ROI16
87
Run141_ROI7
87
Run145_ROI3
74
Run141_ROI19
87
Run141_ROI1
87
Run141_ROI8
87
Run141_ROI17
87
Run145_ROI10
74
Run145_ROI17
71
Run145_ROI11
74
Run103_ROI5
61
Run92_ROI5
53
Run141_ROI15
87
Run145_ROI1
74
Run141_ROI4
87
Run141_ROI23
87
Run141_ROI3
87
Run145_ROI9
74
Run141_ROI14
87
Run145_ROI14
74
Run145_ROI13
74
Run103_ROI6
61
Run92_ROI6
53


## Plotting the histograms of the lowly expressed markers

In [7]:
all_exp = []
for roi in roi_list:
    exp = pd.read_csv(roi / 'exp.csv')
    exp.drop(columns=["cell_id"], inplace=True)
    exp = exp.median(axis=0).to_frame().T
    exp.insert(0, "exp_name", roi.name)
    all_exp.append(exp)
all_exp = pd.concat(all_exp, ignore_index=True)
all_exp_mean = all_exp.iloc[:,1:].median(axis=0)
lowly_antibodies = all_exp_mean[all_exp_mean < 50].index

exp = pd.read_csv(roi / 'exp.csv')
for marker in lowly_antibodies:
    exp[marker].hist(bins=100, color='skyblue', edgecolor='black', log=True)
    plt.title(marker)
    plt.xlabel('Expression')
    plt.show()

KeyError: 'CD123'